# Artificial Intelligence Lab - LAB 07
## Experiment 7: Connect-3 on a 4×4 Board (Minimax vs. Alpha–Beta Pruning)
### Objectives:
1. Implement **Connect-3** on a $4 \times 4$ board with piece gravity.
2. Implement **Plain Minimax Search** and **Minimax with Alpha–Beta Pruning**.
3. Measure and quantify the search space reduction achieved by pruning.

### 1. Board Representation, Gravity, and Win Detection

In [1]:
ROWS = 4
COLS = 4

def create_board():
    return [['-' for _ in range(COLS)] for _ in range(ROWS)]

def display_board(board):
    print("| 1 | 2 | 3 | 4 |")
    print("|:-:|:-:|:-:|:-:|")
    for r in range(ROWS):
        print("| " + " | ".join(board[r]) + " |")

def get_lowest_empty_row(board, col):
    for r in range(ROWS - 1, -1, -1):
        if board[r][col] == '-':
            return r
    return -1

def is_valid_column(board, col):
    if 0 <= col < COLS:
        return board[0][col] == '-'
    return False

def check_win(board, player):
    # Horizontal
    for r in range(ROWS):
        for c in range(COLS - 2):
            if board[r][c] == board[r][c + 1] == board[r][c + 2] == player:
                return True
    # Vertical
    for c in range(COLS):
        for r in range(ROWS - 2):
            if board[r][c] == board[r + 1][c] == board[r + 2][c] == player:
                return True
    # Diagonal \
    for r in range(ROWS - 2):
        for c in range(COLS - 2):
            if board[r][c] == board[r + 1][c + 1] == board[r + 2][c + 2] == player:
                return True
    # Diagonal /
    for r in range(2, ROWS):
        for c in range(COLS - 2):
            if board[r][c] == board[r - 1][c + 1] == board[r - 2][c + 2] == player:
                return True
    return False

def is_board_full(board):
    return all(board[0][c] != '-' for c in range(COLS))

### 2. Search Algorithms: Minimax & Alpha–Beta Pruning

In [2]:
minimax_search_count = 0

def minimax(board, depth, is_maximizing, max_depth=7):
    global minimax_search_count
    minimax_search_count += 1

    if check_win(board, 'O'):
        return 100 - depth
    if check_win(board, 'X'):
        return depth - 100
    if is_board_full(board) or depth == max_depth:
        return 0

    if is_maximizing:
        best_val = float('-inf')
        for c in range(COLS):
            r = get_lowest_empty_row(board, c)
            if r != -1:
                board[r][c] = 'O'
                val = minimax(board, depth + 1, False, max_depth)
                board[r][c] = '-'
                best_val = max(best_val, val)
        return best_val
    else:
        best_val = float('inf')
        for c in range(COLS):
            r = get_lowest_empty_row(board, c)
            if r != -1:
                board[r][c] = 'X'
                val = minimax(board, depth + 1, True, max_depth)
                board[r][c] = '-'
                best_val = min(best_val, val)
        return best_val

def get_best_move_minimax(board, max_depth=7):
    best_val = float('-inf')
    best_col = None
    for c in [2, 1, 3, 0]:
        r = get_lowest_empty_row(board, c)
        if r != -1:
            board[r][c] = 'O'
            val = minimax(board, 1, False, max_depth)
            board[r][c] = '-'
            if val > best_val:
                best_val = val
                best_col = c
    return best_col

alphabeta_search_count = 0

def alphabeta(board, depth, alpha, beta, is_maximizing, max_depth=7):
    global alphabeta_search_count
    alphabeta_search_count += 1

    if check_win(board, 'O'):
        return 100 - depth
    if check_win(board, 'X'):
        return depth - 100
    if is_board_full(board) or depth == max_depth:
        return 0

    if is_maximizing:
        best_val = float('-inf')
        for c in [2, 1, 3, 0]:
            r = get_lowest_empty_row(board, c)
            if r != -1:
                board[r][c] = 'O'
                val = alphabeta(board, depth + 1, alpha, beta, False, max_depth)
                board[r][c] = '-'
                best_val = max(best_val, val)
                alpha = max(alpha, val)
                if beta <= alpha:
                    break
        return best_val
    else:
        best_val = float('inf')
        for c in [2, 1, 3, 0]:
            r = get_lowest_empty_row(board, c)
            if r != -1:
                board[r][c] = 'X'
                val = alphabeta(board, depth + 1, alpha, beta, True, max_depth)
                board[r][c] = '-'
                best_val = min(best_val, val)
                beta = min(beta, val)
                if beta <= alpha:
                    break
        return best_val

def get_best_move_alphabeta(board, max_depth=7):
    best_val = float('-inf')
    best_col = None
    alpha = float('-inf')
    beta = float('inf')
    for c in [2, 1, 3, 0]:
        r = get_lowest_empty_row(board, c)
        if r != -1:
            board[r][c] = 'O'
            val = alphabeta(board, 1, alpha, beta, False, max_depth)
            board[r][c] = '-'
            if val > best_val:
                best_val = val
                best_col = c
            alpha = max(alpha, val)
    return best_col

### 3. Simulation Driver

In [3]:
def play_connect3(algorithm="alphabeta", preset_moves=None, max_depth=7):
    global minimax_search_count, alphabeta_search_count
    if algorithm == "minimax":
        print("---- Connect 3 Using MinMax Algorithm ----")
        minimax_search_count = 0
    else:
        print("---- Connect 3 Using Alpha Beta Pruning ----")
        alphabeta_search_count = 0

    print("User is X")
    print("AI is O")
    print("Here is the empty board:")
    board = create_board()
    display_board(board)

    input_iter = iter(preset_moves) if preset_moves else None

    def get_user_choice():
        if input_iter:
            choice = next(input_iter)
            print(f"User Choice of Drop : {choice}")
            return choice
        raw = input("User Choice of Drop (1-4): ").strip()
        return int(raw)

    while True:
        user_col = get_user_choice() - 1
        ur = get_lowest_empty_row(board, user_col)
        board[ur][user_col] = 'X'
        display_board(board)

        if check_win(board, 'X'):
            print("User Won!")
            break
        if is_board_full(board):
            print("Game Over: Draw!")
            break

        if algorithm == "minimax":
            ai_col = get_best_move_minimax(board, max_depth)
            searches = minimax_search_count
        else:
            ai_col = get_best_move_alphabeta(board, max_depth)
            searches = alphabeta_search_count

        print(f"AI Thinking ... AI Choice : {ai_col + 1}")
        air = get_lowest_empty_row(board, ai_col)
        board[air][ai_col] = 'O'
        display_board(board)

        if check_win(board, 'O'):
            print(f"AI Won.... Number of Searches done in total : {searches:,}")
            break
        if is_board_full(board):
            print(f"Game Over: Draw! Searches done in total : {searches:,}")
            break

### 4. Running Benchmark Comparison

In [4]:
sample_user_drops = [2, 4, 2]

print("=== 1. Plain Minimax Execution ===")
play_connect3(algorithm="minimax", preset_moves=sample_user_drops, max_depth=7)

print("\n=== 2. Alpha-Beta Pruning Execution ===")
play_connect3(algorithm="alphabeta", preset_moves=sample_user_drops, max_depth=7)

mm_count = minimax_search_count
ab_count = alphabeta_search_count
pruned = mm_count - ab_count
reduction_pct = (pruned / mm_count) * 100 if mm_count > 0 else 0

print("\n" + "=" * 60)
print(f"Plain Minimax Total Node Evaluations:   {mm_count:,}")
print(f"Alpha-Beta Pruning Node Evaluations:    {ab_count:,}")
print(f"Search Space Reduction:                 {reduction_pct:.2f}%")
print("=" * 60)

=== 1. Plain Minimax Execution ===
---- Connect 3 Using MinMax Algorithm ----
User is X
AI is O
Here is the empty board:
| 1 | 2 | 3 | 4 |
|:-:|:-:|:-:|:-:|
| - | - | - | - |
| - | - | - | - |
| - | - | - | - |
| - | - | - | - |
User Choice of Drop : 2
| 1 | 2 | 3 | 4 |
|:-:|:-:|:-:|:-:|
| - | - | - | - |
| - | - | - | - |
| - | - | - | - |
| - | X | - | - |
AI Thinking ... AI Choice : 3
| 1 | 2 | 3 | 4 |
|:-:|:-:|:-:|:-:|
| - | - | - | - |
| - | - | - | - |
| - | - | - | - |
| - | X | O | - |
User Choice of Drop : 4
| 1 | 2 | 3 | 4 |
|:-:|:-:|:-:|:-:|
| - | - | - | - |
| - | - | - | - |
| - | - | - | - |
| - | X | O | X |
AI Thinking ... AI Choice : 3
| 1 | 2 | 3 | 4 |
|:-:|:-:|:-:|:-:|
| - | - | - | - |
| - | - | - | - |
| - | - | O | - |
| - | X | O | X |
User Choice of Drop : 2
| 1 | 2 | 3 | 4 |
|:-:|:-:|:-:|:-:|
| - | - | - | - |
| - | - | - | - |
| - | X | O | - |
| - | X | O | X |
AI Thinking ... AI Choice : 3
| 1 | 2 | 3 | 4 |
|:-:|:-:|:-:|:-:|
| - | - | - | - |
| - | - | O | -